# Gráficos das atividades 2, 3 e 4

Usa as tabelas que estão nas pastas `atividade_2`, `atividade_3` e `atividade_4`.

In [ ]:
# python -m pip install pandas matplotlib seaborn xlrd openpyxl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def ler_tabela_ibge(arquivo, nomes_colunas):
    """Lê as tabelas do IBGE pulando o cabeçalho de várias linhas.
    Pega da linha 'Brasil' até a linha 'Distrito Federal'."""
    bruto = pd.read_excel(arquivo, header=None)
    primeira_col = bruto.columns[0]
    inicio = bruto.index[bruto[primeira_col] == "Brasil"][0]
    fim = bruto.index[bruto[primeira_col] == "Distrito Federal"][0]
    df = bruto.loc[inicio:fim, bruto.columns[: len(nomes_colunas)]].copy()
    df.columns = nomes_colunas
    df = df.reset_index(drop=True)
    for c in nomes_colunas[1:]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

REGIOES = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]

## Atividade 2 — Horas que mulheres trabalham em serviços domésticos

In [ ]:
indicador_1 = ler_tabela_ibge(
    "../atividade_2/Tabela 1.1.1.xls",
    ["uf_regiao", "total", "total_branca", "total_preta_parda",
     "homem_branca", "homem_preta_parda", "mulher_branca", "mulher_preta_parda"],
)
regioes = REGIOES

# Barras agrupadas: homens x mulheres por grande região
reg = indicador_1[indicador_1["uf_regiao"].isin(regioes)].copy()
reg["homens"] = (reg["homem_branca"] + reg["homem_preta_parda"]) / 2
reg["mulheres"] = (reg["mulher_branca"] + reg["mulher_preta_parda"]) / 2

reg_long = reg.melt(
    id_vars="uf_regiao",
    value_vars=["homens", "mulheres"],
    var_name="sexo",
    value_name="horas",
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=reg_long, x="uf_regiao", y="horas", hue="sexo", ax=ax)
ax.set_title("Afazeres domésticos por região e sexo")
ax.set_xlabel("Região")
ax.set_ylabel("Horas / semana")
plt.tight_layout()
plt.show()

## Atividade 3 — Formação acadêmica

In [ ]:
tabela3 = ler_tabela_ibge(
    "../atividade_3/Tabela_3_Areas_gerais_de_formacao_na_graduacao.xlsx",
    ["uf_regiao", "total", "total_homens", "total_mulheres",
     "stem_total", "stem_homens", "stem_mulheres",
     "saude_total", "saude_homens", "saude_mulheres"],
)
formacao_total = tabela3[["uf_regiao", "total_homens", "total_mulheres"]]
formacao_stem = tabela3[["uf_regiao", "stem_homens", "stem_mulheres"]].rename(
    columns={"stem_homens": "homens", "stem_mulheres": "mulheres"})
formacao_saude = tabela3[["uf_regiao", "saude_homens", "saude_mulheres"]].rename(
    columns={"saude_homens": "homens", "saude_mulheres": "mulheres"})

# Barras agrupadas: homens x mulheres no Brasil (três recortes)
brasil = pd.DataFrame(
    {
        "area": ["Graduação (total)", "STEM", "Saúde / educação"],
        "homens": [
            formacao_total.loc[formacao_total["uf_regiao"] == "Brasil", "total_homens"].item(),
            formacao_stem.loc[formacao_stem["uf_regiao"] == "Brasil", "homens"].item(),
            formacao_saude.loc[formacao_saude["uf_regiao"] == "Brasil", "homens"].item(),
        ],
        "mulheres": [
            formacao_total.loc[formacao_total["uf_regiao"] == "Brasil", "total_mulheres"].item(),
            formacao_stem.loc[formacao_stem["uf_regiao"] == "Brasil", "mulheres"].item(),
            formacao_saude.loc[formacao_saude["uf_regiao"] == "Brasil", "mulheres"].item(),
        ],
    }
)
brasil_long = brasil.melt(id_vars="area", var_name="sexo", value_name="pessoas")

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=brasil_long, x="area", y="pessoas", hue="sexo", ax=ax)
ax.set_title("Brasil: pessoas com graduação, por área e sexo")
ax.set_xlabel("Área")
ax.set_ylabel("Pessoas")
plt.tight_layout()
plt.show()

## Atividade 4 — População 2022 por UF

In [ ]:
pop = pd.read_excel("../atividade_4/CD2022_Populacao_2010_Compatibilizada_20231222.xlsx", skiprows=2)
pop = pop.drop(columns=pop.columns[0])
pop.columns = ["UF", "COD_UF", "COD_MUNIC", "MUNICIPIO", "2010_sinopse", "2010", "2022"]
pop = pop.dropna(subset=["UF", "2022"])
pop_estado = pop.groupby("UF")[["2010", "2022"]].sum().reset_index()

fig, ax = plt.subplots(figsize=(9, 8))
ordem = pop_estado.sort_values("2022", ascending=False)
sns.barplot(data=ordem, y="UF", x="2022", ax=ax, color="steelblue")
ax.set_title("População 2022 por UF")
ax.set_xlabel("Habitantes")
ax.set_ylabel("UF")
plt.tight_layout()
plt.show()

In [ ]:
# Extra: 2010 x 2022 com melt
pop_long = pop_estado.melt(id_vars="UF", value_vars=["2010", "2022"], var_name="ano", value_name="populacao")
ordem_uf = pop_estado.sort_values("2022", ascending=False)["UF"]

fig, ax = plt.subplots(figsize=(9, 9))
sns.barplot(data=pop_long, y="UF", x="populacao", hue="ano", order=ordem_uf, ax=ax)
ax.set_title("População por UF: 2010 x 2022")
ax.set_xlabel("Habitantes")
ax.set_ylabel("UF")
plt.tight_layout()
plt.show()